# Algorytmika i matematyka uczenia maszynowego 
## Laboratorium 11

### Zadanie 1

Zaimplementuj systemu rekomendacji filmów w oparciu o indeks Jaccarda. System ma za zadanie zwrócić listę filmów sugerowany dla podanego użytownika.

Dane zostały pobrane z serwisu Kaggle z https://www.kaggle.com/datasets/gargmanas/movierecommenderdataset

Zbiór zawiera dwa pliki:
- `movies.csv` lista filmów wraz z ich identyfikatorami
- `ratings.csv` lista ocen filmów przez użytkowników

**Zadanie:**
* Wczytaj oba pliki.
* Zamień wszystkie oceny użytkownika na wartość 1 (zastosuj próg okreśjący czy film się podobał czy nie np. 3).
* Stwórz macierz ocen użytkowników w której wierszach będą użytkownicy, a w kolumnach filmy. Wartość w macierzy jest flagą mówiącą czy użytkownikowi film się podobał czy nie. 
* Wypełnij brakujące wartości zerami.
* Utwórz macierz podobieństwa Jaccarda pomiędzy użytkownikami (każdy z każdym).
    - Możesz wykorzystać funkcję [jaccard](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.jaccard.html) z biblioteki scipy.
* Zaimplementuj funkcję która dla podanego użytkownika zwróci listę sugerowanych filmów.
    - Funkcja powinna zwrócić listę filmów które nie były ocenione przez użytkownika, a które są rekomendowane dla niego.



In [3]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform


movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

ratings['rating'] = (ratings['rating'] > 2.5).astype(int)

movies_id = movies['movieId'].unique()
users_id = ratings['userId'].unique()
user_ratings = np.zeros((len(users_id), len(movies_id)))

movie_to_index = {movie_id: idx for idx, movie_id in enumerate(movies_id)}
user_to_index = {user_id: idx for idx, user_id in enumerate(users_id)}

for _, row in ratings.iterrows():
    if row['rating'] == 1:
        user_idx = user_to_index[row['userId']]
        movie_idx = movie_to_index[row['movieId']]
        user_ratings[user_idx][movie_idx] = 1

jaccard_distances = pdist(user_ratings, metric='jaccard')

jaccard_similarity = 1 - squareform(jaccard_distances)
print(jaccard_similarity)



[[1.         0.00796813 0.01244813 ... 0.13940256 0.03543307 0.0516325 ]
 [0.00796813 1.         0.         ... 0.01006711 0.01587302 0.01454234]
 [0.01244813 0.         1.         ... 0.00168919 0.         0.00341006]
 ...
 [0.13940256 0.01006711 0.00168919 ... 1.         0.03030303 0.15831663]
 [0.03543307 0.01587302 0.         ... 0.03030303 1.         0.0084317 ]
 [0.0516325  0.01454234 0.00341006 ... 0.15831663 0.0084317  1.        ]]


In [ ]:
def recommend_movies(user_id, user_ratings, user_to_index, movies_id, jaccard_similarity, count=5):
    user_idx = user_to_index[user_id]
    
    similarities = jaccard_similarity[user_idx]
    similar_users_idx = np.argsort(-similarities)
    similar_users_idx = [idx for idx in similar_users_idx if idx != user_idx][:5]
    
    similar_users_ratings = user_ratings[similar_users_idx]
    scores = np.sum(similar_users_ratings, axis=0)
    
    unseen_by_user = user_ratings[user_idx] == 0
    
    recommended_indices = np.argsort(-scores * unseen_by_user)
    recommended_movie_ids = [movies_id[idx] for idx in recommended_indices if unseen_by_user[idx] and scores[idx] > 0]
    
    return recommended_movie_ids[:5]

user_id = 15
# watched_movie_indices = np.where(user_ratings[user_to_index[user_id]] == 1)[0]
# watched_movie_ids = [movies_id[idx] for idx in watched_movie_indices]
# watched_titles = movies[movies['movieId'].isin(watched_movie_ids)]['title'].tolist()
# print(f"Filmy obejrzane przez Usera o ID: {user_id}:")
# for title in watched_titles:
#     print(title)

recommended = recommend_movies(user_id, user_ratings, user_to_index, movies_id, jaccard_similarity)
recommended_titles = movies[movies['movieId'].isin(recommended)]['title'].tolist()
print(f"Tytuły polecanych filmów dla Usera o ID: {user_id}")
for title in recommended_titles:
    print(title)

Filmy obejrzane przez Usera o ID: 15:
Seven (a.k.a. Se7en) (1995)
Junior (1994)
Star Wars: Episode IV - A New Hope (1977)
Léon: The Professional (a.k.a. The Professional) (Léon) (1994)
Pulp Fiction (1994)
Shawshank Redemption, The (1994)
Forrest Gump (1994)
Lion King, The (1994)
Schindler's List (1993)
Aladdin (1992)
Terminator 2: Judgment Day (1991)
Independence Day (a.k.a. ID4) (1996)
Godfather, The (1972)
Star Wars: Episode V - The Empire Strikes Back (1980)
Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)
Aliens (1986)
Star Wars: Episode VI - Return of the Jedi (1983)
Alien (1979)
Terminator, The (1984)
Groundhog Day (1993)
Back to the Future (1985)
Nightmare on Elm Street, A (1984)
Fifth Element, The (1997)
Gattaca (1997)
Lethal Weapon 2 (1989)
Back to the Future Part II (1989)
Back to the Future Part III (1990)
Saving Private Ryan (1998)
Little Mermaid, The (1989)
Gods Must Be Crazy, The (1980)
American History X (1998)
Matrix, The (1999)
Sixth Sense



### Zadanie 2


Algorytm MinHash na przykładzie wykrywania plagiatów

Wykonaj kolejno następujące kroki:

1. Pobierz 8 akapitów tekstu (nie za krótkich), każdy o różnej tematyce (mogą być np. z różnych haseł Wikipedii), trzymaj się jednego języka (np. PL lub ENG). Wklej je do jednego pliku tekstowego, z linią wolną jako separatorem.

2. Skopiuj wybrane 2-3 akapity i ręcznie nieco zmodyfikuj.

> Przykład (z hasła https://pl.wikipedia.org/wiki/Fryderyk_Chopin):

```Jest uważany za jednego z najwybitniejszych kompozytorów romantycznych, a także za jednego z najważniejszych polskich kompozytorów w historii. Był jednym z najsłynniejszych pianistów swoich czasów, często nazywany poetą fortepianu. Elementem charakterystycznym dla utworów Chopina jest pogłębiona ekspresja oraz czerpanie z wzorców stylistycznych polskiej muzyki ludowej.```

↓↓↓ Zmieniono na ↓↓↓

```Jest uważany za jednego z najwybitniejszych kompozytorów romantycznych, a także za jednego z najważniejszych kompozytorów polskich w historii.  Był jednym  z najsłynniejszych pianistów swoich czasów, często nazywany poetą fortepianu! Elementem charakterystycznym dla utworów Fryderyka Chopina jest pogłębiona ekspresja oraz czerpanie z wzorców polskiej muzyki ludowej.```

Otrzymasz zatem w pliku tekstowym 10 lub 11 akapitów tekstu (kolejność dowolna, te „splagiatowane” nie muszą być na końcu).

3. Z poziomu skryptu: wczytaj wszystkie akapity z pliku. Zbuduj 100 "losowych" funkcji haszujących.

> Sugestia: funkcją "bazową" jest po prostu `hash(...)`. Zakładamy 64-bitową wersję Pythona 3.x, wtedy `hash(...)` jest 64-bitowy.

Na liście seeds umieszczamy 100 losowych liczb 64-bitowych. Aby obliczyć $i$-ty hash dla ciągu 
należy wykonać `hash(s) ^ seeds[i]` (użycie operatora XOR).

4. Przyjmij niewielką wartość $Q$ (np. 15) i dla każdego akapitu
    - oznacz jego długość przez $n$,
    - dla każdej ze 100 funkcji haszujących policz hasza w przesuwnym oknie tekstu o długości $Q$ znaków (czyli łącznie mamy $n - Q + 1$ wartości hasza); zapamiętaj MINIMUM z tych $n - Q + 1$
 wartości.
Na wyjściu mamy zatem (dla 11 akapitów) 11 * 100 wartości haszy.

5. Rozważ pary akapitów "każdy z każdym". Jeśli dla danej pary co najmniej (np.) 30 haszy jest wspólnych, to uważamy akapity za podobne (być może plagiat) i wyświetlamy na ekranie.

6. Wyświetl czas obliczeń (powinien wynosić mniej niż 0.5s).

7. Poeksperymentuj z liczbą użytych funkcji haszujących, wartością, stopniem modyfikacji oryginalnych akapitów tekstu, progiem detekcji akapitów podobnych.

In [1]:
import random
import time

with open("teksty.txt", "r", encoding="utf-8") as f:
    paragraphs = [line.strip() for line in f.read().split('\n\n') if line.strip()]
print(f"Liczba wczytanych akapitów: {len(paragraphs)}")

num_hashes = 100
Q = 10

seeds = [random.getrandbits(64) for _ in range(num_hashes)]

start = time.time()

minhash_signatures = []
for paragraph in paragraphs:
    n = len(paragraph)
    mins = []
    for i in range(num_hashes):
        hashes = [
            hash(paragraph[j :j + Q]) ^ seeds[i]
            for j in range(n - Q + 1)
        ] if n >= Q else [hash(paragraph) ^ seeds[i]]
        mins.append(min(hashes))
    minhash_signatures.append(mins)

print(f"Czas obliczeń: {time.time() - start:.3f}s")

threshold = 5
similar_pairs = []

for i in range(len(minhash_signatures)):
    for j in range(i + 1, len(minhash_signatures)):
        common_hashes = sum(
            1 for a, b in zip(minhash_signatures[i], minhash_signatures[j]) if a == b
        )
        if common_hashes >= threshold:
            similar_pairs.append((i, j, common_hashes))

print("Znalezione podobne akapity:")
for i, j, common in similar_pairs:
    print(f"Akapity: {i} <-> {j}: {common} wspólnych hashy")
    print(paragraphs[i])
    print(paragraphs[j])
    print()


Liczba wczytanych akapitów: 11
Czas obliczeń: 0.091s
Znalezione podobne akapity:
Akapity: 0 <-> 2: 6 wspólnych hashy
Słoń afrykański to największy współcześnie żyjący ssak lądowy. Charakteryzuje się dużymi uszami, które pomagają mu w regulacji temperatury ciała. Słonie afrykańskie żyją w stadach prowadzonych przez samice i odgrywają kluczową rolę w ekosystemach sawann i lasów.
Słoń afrykański jest największym ssakiem lądowym na świecie. Jego wyjątkowo duże uszy pomagają mu schładzać organizm podczas upałów. Stada słoni afrykańskich prowadzone są przez doświadczone samice, a same zwierzęta mają ogromny wpływ na środowisko sawann.

Akapity: 1 <-> 3: 17 wspólnych hashy
Internet to globalna sieć komputerowa, która umożliwia wymianę informacji na całym świecie. Powstał w latach 60. XX wieku jako projekt wojskowy, a obecnie jest nieodłącznym elementem życia codziennego, oferując dostęp do wiedzy, komunikacji i rozrywki.
Internet, będący globalną siecią komputerową, umożliwia szybki przepływ 